In [1]:
import os
import json
import shutil
from PIL import Image
from tqdm import tqdm

# --- CONFIGURATION ---

# 1. Path to the root of your xView dataset.
#    This directory should contain the 'xView_train.geojson' file and the folder with .tif images.
XVIEW_ROOT_PATH = '/caa/Homes01/mburges/datasets/xView'

# 2. Path to the folder where you want to save all the new images.
OUTPUT_IMAGE_DIR = '/caa/Homes01/mburges/datasets/xView/images'

# 3. Path where you want to save the final COCO JSON annotation file.
OUTPUT_JSON_PATH = '/caa/Homes01/mburges/datasets/xView/annotations/xView_coco.json'

# --- CONSTANTS ---

# Assumes the source GeoJSON file is named 'xView_train.geojson' and is in the root path.
SOURCE_GEOJSON_PATH = os.path.join(XVIEW_ROOT_PATH, 'xView_train.geojson')

# Assumes the source .tif images are located directly in the root path.
# If they are in a subfolder (e.g., 'train_images'), change this to:
SOURCE_IMAGE_DIR = os.path.join(XVIEW_ROOT_PATH, 'train_images')
# SOURCE_IMAGE_DIR = XVIEW_ROOT_PATH

# xView classes 11-94 to 0-59
xview_class2index = [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, 0, 1, 2, -1, 3, -1, 4, 5, 6, 7, 8, -1, 9, 10, 11,
                    12, 13, 14, 15, -1, -1, 16, 17, 18, 19, 20, 21, 22, -1, 23, 24, 25, -1, 26, 27, -1, 28, -1,
                    29, 30, 31, 32, 33, 34, 35, 36, 37, -1, 38, 39, 40, 41, 42, 43, 44, 45, -1, -1, -1, -1, 46,
                    47, 48, 49, -1, 50, 51, -1, 52, -1, -1, -1, 53, 54, -1, 55, -1, -1, 56, -1, 57, -1, 58, 59]

names = ['Fixed-wing Aircraft', 'Small Aircraft', 'Cargo Plane', 'Helicopter', 'Passenger Vehicle', 'Small Car', 'Bus',
        'Pickup Truck', 'Utility Truck', 'Truck', 'Cargo Truck', 'Truck w/Box', 'Truck Tractor', 'Trailer',
        'Truck w/Flatbed', 'Truck w/Liquid', 'Crane Truck', 'Railway Vehicle', 'Passenger Car', 'Cargo Car',
        'Flat Car', 'Tank car', 'Locomotive', 'Maritime Vessel', 'Motorboat', 'Sailboat', 'Tugboat', 'Barge',
        'Fishing Vessel', 'Ferry', 'Yacht', 'Container Ship', 'Oil Tanker', 'Engineering Vehicle', 'Tower crane',
        'Container Crane', 'Reach Stacker', 'Straddle Carrier', 'Mobile Crane', 'Dump Truck', 'Haul Truck',
        'Scraper/Tractor', 'Front loader/Bulldozer', 'Excavator', 'Cement Mixer', 'Ground Grader', 'Hut/Tent', 'Shed',
        'Building', 'Aircraft Hangar', 'Damaged Building', 'Facility', 'Construction Site', 'Vehicle Lot', 'Helipad',
        'Storage Tank', 'Shipping container lot', 'Shipping Container', 'Pylon', 'Tower']


# A mapping from the original xView class IDs to their names.
XVIEW_CLASS_MAP = {
    i + 11: name for i, name in enumerate(names) if xview_class2index[i + 11] != -1
}


def convert_xview_to_coco():
    """
    Main function to perform the dataset conversion.
    """
    # --- 1. Setup Phase ---
    print("🚀 Starting conversion from xView to COCO format...")
    print("Setting up output directories...")
    os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)
    os.makedirs(os.path.dirname(OUTPUT_JSON_PATH), exist_ok=True)

    # --- 2. Category Creation ---
    print("Creating COCO categories...")
    categories = []
    # Sort by original xView ID to ensure a consistent mapping to new COCO IDs
    sorted_xview_ids = sorted(XVIEW_CLASS_MAP.keys())
    # Create a mapping from xView ID -> COCO ID (which will be 1-based)
    xview_id_to_coco_id = {xview_id: i + 1 for i, xview_id in enumerate(sorted_xview_ids)}

    for xview_id in sorted_xview_ids:
        categories.append({
            "id": xview_id_to_coco_id[xview_id],
            "name": XVIEW_CLASS_MAP[xview_id],
            "supercategory": "object",
        })

    # --- 3. Data Loading and Grouping ---
    print(f"Loading GeoJSON file from: {SOURCE_GEOJSON_PATH}")
    with open(SOURCE_GEOJSON_PATH) as f:
        geojson_data = json.load(f)

    # Group all annotations by their corresponding image filename
    annotations_by_image = {}
    print("Grouping annotations by image...")
    for feature in tqdm(geojson_data['features'], desc="Processing GeoJSON Features"):
        props = feature['properties']
        image_filename = props['image_id']
        if image_filename not in annotations_by_image:
            annotations_by_image[image_filename] = []
        annotations_by_image[image_filename].append(props)

    # --- 4. COCO JSON Generation ---
    coco_output = {
        "info": {"description": "xView Dataset converted to COCO format"},
        "licenses": [{"url": "https://challenge.xviewdataset.org/data-license", "id": 1, "name": "xView License"}],
        "images": [],
        "annotations": [],
        "categories": categories
    }

    image_id_counter = 1
    annotation_id_counter = 1

    print("\nProcessing images and creating annotations...")
    for image_filename, annotations in tqdm(annotations_by_image.items(), desc="Converting Images"):
        # --- Image Handling ---
        source_image_path = os.path.join(SOURCE_IMAGE_DIR, image_filename)
        if not os.path.exists(source_image_path):
            print(f"⚠️ Warning: Image file not found, skipping: {source_image_path}")
            continue

        # Copy the image to the output directory
        shutil.copyfile(source_image_path, os.path.join(OUTPUT_IMAGE_DIR, image_filename))

        # Get image dimensions
        try:
            with Image.open(source_image_path) as img:
                width, height = img.size
        except Exception as e:
            print(f"❌ Error opening image {source_image_path}: {e}. Skipping this image.")
            continue
        
        # Create the image record for the COCO file
        image_record = {
            "id": image_id_counter,
            "file_name": image_filename,
            "width": width,
            "height": height,
            "license": 1,
        }
        coco_output['images'].append(image_record)
        
        # --- Annotation Handling ---
        for ann in annotations:
            try:
                coords_str = ann['bounds_imcoords']
                # Clean the string by removing brackets and spaces, then split by comma
                cleaned_coords_str = coords_str.strip().replace('[', '').replace(']', '')
                parts = cleaned_coords_str.split(',')
                # Convert all parts to integers
                xmin, ymin, xmax, ymax = [int(p.strip()) for p in parts]
            except (json.JSONDecodeError, ValueError):
                print(f"⚠️ Warning: Could not parse bbox '{coords_str}' for image {image_filename}. Skipping annotation.")
                continue

            # Convert to COCO format [xmin, ymin, width, height]
            bbox_width = xmax - xmin
            bbox_height = ymax - ymin
            
            if bbox_width <= 0 or bbox_height <= 0:
                continue

            coco_bbox = [xmin, ymin, bbox_width, bbox_height]
            
            xview_type_id = int(ann['type_id'])
            if xview_type_id not in xview_id_to_coco_id:
                continue # Skip if the class is not in our defined map

            # Create the annotation record
            annotation_record = {
                "id": annotation_id_counter,
                "image_id": image_id_counter,
                "category_id": xview_id_to_coco_id[xview_type_id],
                "bbox": coco_bbox,
                "area": float(bbox_width * bbox_height),
                "iscrowd": 0,
                "segmentation": [], # Bounding boxes only, no segmentation
            }
            coco_output['annotations'].append(annotation_record)
            annotation_id_counter += 1
            
        image_id_counter += 1

    # --- 5. Save Final JSON ---
    print(f"\n💾 Saving final COCO JSON file to: {OUTPUT_JSON_PATH}")
    with open(OUTPUT_JSON_PATH, 'w') as f:
        json.dump(coco_output, f, indent=4)
        
    print("\n🎉 Conversion complete!")
    print(f"📊 Total images processed: {len(coco_output['images'])}")
    print(f"📦 Total annotations created: {len(coco_output['annotations'])}")


if __name__ == "__main__":
    # Perform checks before running the main conversion function
    if not os.path.isfile(SOURCE_GEOJSON_PATH):
        print(f"❌ Error: GeoJSON file not found at {SOURCE_GEOJSON_PATH}")
        print("👉 Please check that your 'XVIEW_ROOT_PATH' is correct and contains 'xView_train.geojson'.")
    elif not os.path.isdir(SOURCE_IMAGE_DIR):
        print(f"❌ Error: Source image directory not found at {SOURCE_IMAGE_DIR}")
        print("👉 Please check your 'SOURCE_IMAGE_DIR' path.")
    else:
        convert_xview_to_coco()

🚀 Starting conversion from xView to COCO format...
Setting up output directories...
Creating COCO categories...
Loading GeoJSON file from: /caa/Homes01/mburges/datasets/xView/xView_train.geojson
Grouping annotations by image...


Processing GeoJSON Features: 100%|██████████| 601937/601937 [00:00<00:00, 2133788.23it/s]



Processing images and creating annotations...


Converting Images:  75%|███████▌  | 638/847 [00:26<00:06, 30.30it/s]

⚠️ Warning: Image file not found, skipping: /caa/Homes01/mburges/datasets/xView/train_images/1395.tif


Converting Images: 100%|██████████| 847/847 [00:35<00:00, 23.96it/s]



💾 Saving final COCO JSON file to: /caa/Homes01/mburges/datasets/xView/annotations/xView_coco.json

🎉 Conversion complete!
📊 Total images processed: 846
📦 Total annotations created: 269138
